Base corregida para entrenar la red

Parte de datos_imputados.csv y corrige sus tres problemas principales, más que ndad se enfoca en recupera los datos reales de 2024 (anteriormente estaban en columnas con unidades y el imputador no los leyó). Usa siempre el dato observado cuando existe (ya limpio con los criterios de la Etapa 2) y deja lo imputado solo donde faltaba. y por último corrige valores físicamente imposibles. Es que antes en lo imputado (negativos, RH > 100 %, PM2.5 > PM10).



In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA = Path("..") / "data" / "procesados"
VARS = ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PM2.5", "PRS",
        "RAINF", "RH", "SO2", "SR", "TOUT", "WSR", "WDR"]

im = pd.read_csv(DATA / "datos_imputados.csv")
ni = pd.read_csv(DATA / "datos_no_imputados.csv", low_memory=False)
for c in [c for c in ni.columns if " (" in c]:          # recupera 2024
    ni[c.split(" (")[0]] = ni[c.split(" (")[0]].fillna(ni[c])
for df in (im, ni):
    df["Fecha y hora"] = pd.to_datetime(df["Fecha y hora"])

# Aquí la indicación de la lluvia
ESTACIONES_LLUVIA_X20 = ["CE", "NE", "NO2", "NO3", "NTE2", "SE", "SO"]
for df in (im, ni):
    df.loc[df["Estacion"].isin(ESTACIONES_LLUVIA_X20), "RAINF"] *= 20
print(im.shape, ni.shape)

(831095, 17) (830904, 33)


Aquí Limpia de los datos observados (osea junta la Etapa 2 y valores meteorológicos imposibles)

In [2]:
obs = ni[["Estacion", "Fecha y hora"] + VARS].copy()
NO_NEG = ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PM2.5", "PRS", "RAINF", "RH", "SO2", "SR", "WSR", "WDR"]
obs[NO_NEG] = obs[NO_NEG].mask(obs[NO_NEG] < 0)
obs.loc[obs["RH"] > 105, "RH"] = np.nan
obs["RH"] = obs["RH"].clip(upper=100)
pm = obs["PM2.5"] > obs["PM10"]
obs.loc[pm, ["PM2.5", "PM10"]] = np.nan
obs.loc[(obs["TOUT"] < -10) | (obs["TOUT"] > 50), "TOUT"] = np.nan
obs.loc[obs["PM10"] >= 999, "PM10"] = np.nan
obs.loc[obs["SR"] > 1.5, "SR"] = np.nan
obs.loc[(obs["PRS"] < 650) | (obs["PRS"] > 780), "PRS"] = np.nan
obs.loc[obs["WSR"] > 150, "WSR"] = np.nan
obs.loc[obs["RAINF"] > 100, "RAINF"] = np.nan
obs.loc[obs["WDR"] > 360, "WDR"] = np.nan

Y pues combiné no se si esta bien o no pero se combina lo bservado cuando existe, imputa solo donde faltaba

In [3]:
crudo = ni[["Estacion", "Fecha y hora"] + VARS].rename(columns={v: v + "_crudo" for v in VARS})
m = (im.merge(obs, on=["Estacion", "Fecha y hora"], how="left", suffixes=("", "_obs"))
       .merge(crudo, on=["Estacion", "Fecha y hora"], how="left"))
corregida = im[["Estacion", "Fecha y hora"]].copy()
mascara = im[["Estacion", "Fecha y hora"]].copy()
invalidados = {}
for v in VARS:
    hay_dato = m[v + "_obs"].notna()
    # Si había un dato crudo pero la limpieza lo invalidó, la base imputada trae ese mismo valor
    # error: (la imputación conservó los observados). Esos casos se dejan vacíos para rellenarlos abajo.
    invalido = m[v + "_crudo"].notna() & ~hay_dato
    invalidados[v] = int(invalido.sum())
    corregida[v] = m[v + "_obs"].where(hay_dato, m[v].where(~invalido))
    mascara[v] = hay_dato.astype(int)
print("Valores inválidos que se rellenan:", {k: n for k, n in invalidados.items() if n})

# Relleno de los inválidos: interpolación lineal dentro de cada estación (hasta 6 h)
# y, si no alcanza, la mediana de la estación
corregida = corregida.sort_values(["Estacion", "Fecha y hora"])
for v in VARS:
    g = corregida.groupby("Estacion")[v]
    corregida[v] = g.transform(lambda s: s.interpolate(limit=6, limit_area="inside"))
    corregida[v] = corregida[v].fillna(corregida.groupby("Estacion")[v].transform("median"))
corregida = corregida.sort_index()

# Límites físicos para los valores que siguen siendo imputados
POS = ["CO", "NO", "NO2", "NOX", "O3", "PM10", "PM2.5", "RAINF", "SO2", "SR", "WSR"]
corregida[POS] = corregida[POS].clip(lower=0)
corregida["RH"] = corregida["RH"].clip(0, 100)
corregida["WDR"] = corregida["WDR"].clip(0, 360)
corregida["SR"] = corregida["SR"].clip(upper=1.5)
mal = corregida["PM2.5"] > corregida["PM10"]
corregida.loc[mal, "PM2.5"] = corregida.loc[mal, "PM10"]
print("Faltantes en la base corregida:", int(corregida[VARS].isna().sum().sum()))

Valores inválidos que se rellenan: {'CO': 1, 'O3': 1, 'PM10': 43, 'PM2.5': 39, 'PRS': 3, 'RAINF': 373, 'RH': 21, 'SR': 801, 'TOUT': 430, 'WSR': 97}
Faltantes en la base corregida: 0


Comprobaciones

In [4]:
año = corregida["Fecha y hora"].dt.year
pd.DataFrame({
    "O3 > 38 (imputada)":  im[im["O3"] > 38].groupby(año).size(),
    "O3 > 38 (corregida)": corregida[corregida["O3"] > 38].groupby(año).size(),
    "% O3 observado":      (mascara["O3"].groupby(año).mean() * 100).round(1),
})

,O3 > 38 (imputada),O3 > 38 (corregida),% O3 observado
Fecha y hora,,,
2020,10151.0,10151,39.4
2021,24357.0,24357,86.9
2022,25773.0,25773,96.3
2023,25668.0,25668,95.0
2024,NaN,31737,95.1
2025,33095.0,33095,93.4
2026,19165.0,19165,97.3


In [5]:
pd.Series({
    "Negativos": int((corregida[POS] < 0).sum().sum()),
    "RH > 100": int((corregida["RH"] > 100).sum()),
    "PM2.5 > PM10": int((corregida["PM2.5"] > corregida["PM10"]).sum()),
    "PM10 >= 999": int((corregida["PM10"] >= 999).sum()),
    "Temperatura fuera de [-10, 50]": int(((corregida["TOUT"] < -10) | (corregida["TOUT"] > 50)).sum()),
    "SR > 1.5": int((corregida["SR"] > 1.5).sum()),
})

Negativos                         0
RH > 100                          0
PM2.5 > PM10                      0
PM10 >= 999                       0
Temperatura fuera de [-10, 50]    0
SR > 1.5                          0
dtype: int64

In [6]:
corregida.to_csv(DATA / "datos_corregidos.csv", index=False)
mascara.to_csv(DATA / "mascara_observados.csv", index=False)
print("Guardados:", corregida.shape, mascara.shape)

Guardados: (831095, 17) (831095, 17)
